
# Scripted Trade with Heston

Demonstrate using the Heston model 
- calibration to DAX, STOXX50 and S&P500 options
- pricing: Vanilla Equity Option, Barrier Option, Rainbow Option

Prerequisites:
- Python 3
- pandas, matplotlib
- ORE Python (first 2026 release)

### Run ORE

In [ ]:
from ORE import *
import sys, time, math
sys.path.append('..')
import utilities

params = Parameters()
params.fromFile("Input/ore_heston.xml")
#params.fromFile("Input/ore_black.xml")

ore = OREApp(params)
ore.run()

utilities.checkErrorsAndRunTime(ore)

### NPV report

In [ ]:
report = ore.getReport("npv")
utilities.format_report(report)

### Calibration report

In [ ]:
import pandas as pd

df = pd.read_csv('Output/assetmodelcalibration.csv')

pd.set_option('display.expand_frame_repr', False)
print(df)

### Filter one index and expiry

In [ ]:
trade = "5:RainbowOption"
index = "EQ-RIC:.STOXX50E"
#index = "EQ-RIC:.GDAXI"
#index = "EQ-RIC:.SPX"
expiry = "3M"
df = df[df["#TradeID"] == trade]
df = df[df["Index"] == index]
df = df[df["Expiry"] == expiry]
print(df)

### Plot smile section

In [ ]:
x = df["Moneyness"]
y1 = df["MarketVol"]
y2 = df["ModelVol"]

import matplotlib.pyplot as plt

plt.plot(x, y1, marker="o", linewidth=2, label='Market Vol')
plt.plot(x, y2, marker="3", linewidth=2, label='Model Vol')

plt.xlabel("Moneyness") 
plt.ylabel("Implied Vol") 
plt.title(index + ' Expiry 3M')
plt.legend()

plt.show()

### Get some additional results

In [ ]:
res = pd.read_csv('Output/additional_results.csv')
pd.set_option('display.min_rows', 50)
pd.set_option('display.max_rows', 50)

# filter the trade
res = res[res["#TradeId"] == trade]
print(res[res["ResultId"].str.contains("Heston.Index")], "\n")
print(res[res["ResultId"].str.contains("Heston.Forward")], "\n")
#print(res[res["ResultId"].str.contains("Heston.theta")], "\n")
#print(res[res["ResultId"].str.contains("Heston.kappa")], "\n")
#print(res[res["ResultId"].str.contains("Heston.sigma")], "\n")
#print(res[res["ResultId"].str.contains("Heston.rho")], "\n")
#print(res[res["ResultId"].str.contains("Heston.v0")], "\n")



### Get path data 

In [ ]:
import numpy as np
from math import sqrt, exp, log

date = "2023-09-05"

res = pd.read_csv('Output/assetmodelpaths.csv')
res = res[res["#TradeId"] == trade]
res = res[res["Index"] == index]
res = res[res["Date"] == date]
# extract the sample values
data = res["Value"].to_numpy(dtype="float32")

# convert to log ( f(t) / forward )
forwardPrice = 4371.906927 
data = np.log(data/forwardPrice)

#print(res)
print ("xdata:", data, "\n")

print("length:", len(data))
print("mean:  ", np.mean(data))
print("stdev: ", math.sqrt(np.var(data)))
      

In [ ]:
plt.hist(data, density=False, bins=50)
plt.xlabel("Index Value") 
plt.ylabel("Frequency") 
plt.title("Spot Distribution for Index 0 and Date 0")
#plt.legend()
plt.show()